# Prophet Baseline

Đánh giá Prophet trên toàn bộ 100 series (Store × Product), theo cùng convention với `run_naive.py` và `run_lstm2.py`.

In [3]:
import numpy as np
import pandas as pd
import warnings
import sys, os
from prophet import Prophet

warnings.filterwarnings('ignore')
sys.path.append('..')

In [4]:
import tensorflow as tf
import sys, os
import glob
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

tf.random.set_seed(42)
np.random.seed(42)

if os.path.exists("/kaggle/input"):

    matches = glob.glob(
        "/kaggle/input/**/sales_data.csv",
        recursive=True
    )

    DATA_PATH = matches[0] if matches else "/kaggle/input/sales_data.csv"
    RESULT_DIR = "/kaggle/working"

else:

    DATA_PATH  = "/Users/P837032/Daily/Model/dataset/sales_data.csv"
    RESULT_DIR = "/Users/P837032/Daily/Model/result"


print("DATA_PATH:", DATA_PATH)

TARGET     = 'Units Sold'
TRAIN_END  = '2023-06-30'
VAL_END    = '2023-10-31'
HORIZONS   = [7, 14, 28]
LOOKBACK   = 30
LAG        = 7

E0000 00:00:1773887511.469979      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773887511.545790      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773887512.185531      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773887512.185596      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773887512.185599      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773887512.185602      55 computation_placer.cc:177] computation placer already registered. Please check linka

DATA_PATH: /kaggle/input/datasets/thaonngyn/retail-data/sales_data.csv


In [5]:
df = pd.read_csv(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Store ID', 'Product ID', 'Date'])

series_dict = {
    f"{store}_{product}": grp.set_index('Date')[TARGET]
    for (store, product), grp in df.groupby(['Store ID', 'Product ID'])
}
series_ids = sorted(series_dict.keys())
print(f'Total series: {len(series_ids)}')

Total series: 100


In [6]:
def prophet_fn(train_dates, train_values, horizon):
    df_fit = pd.DataFrame({'ds': train_dates, 'y': train_values.astype(float)})
    m = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=False)
    m.fit(df_fit)
    future = m.make_future_dataframe(periods=horizon, freq='D', include_history=False)
    fc = m.predict(future)['yhat'].values
    return np.clip(fc, 0, None)


def rolling_eval(series, split_end, horizon):
    eval_start = pd.Timestamp(split_end) + pd.Timedelta(days=1)
    eval_end   = series.index.max()
    all_fc, all_ac = [], []
    t = eval_start
    while t + pd.Timedelta(days=horizon - 1) <= eval_end:
        train_s = series[:t - pd.Timedelta(days=1)]
        actual  = series[t: t + pd.Timedelta(days=horizon - 1)].values.astype(float)
        if len(train_s) < 2 or len(actual) < horizon:
            t += pd.Timedelta(days=horizon); continue
        all_fc.append(prophet_fn(train_s.index, train_s.values, horizon))
        all_ac.append(actual)
        t += pd.Timedelta(days=horizon)
    if not all_fc:
        return {k: np.nan for k in ['mse', 'rmse', 'mae', 'mape']}
    fc_arr = np.array(all_fc)
    ac_arr = np.array(all_ac)
    return {
        'mse' : np.mean((fc_arr - ac_arr) ** 2),
        'rmse': np.sqrt(np.mean((fc_arr - ac_arr) ** 2)),
        'mae' : np.mean(np.abs(fc_arr - ac_arr)),
        'mape': np.mean(np.abs((fc_arr - ac_arr) / (ac_arr + 1e-8))) * 100,
    }

In [7]:
os.makedirs(RESULT_DIR, exist_ok=True)
results = []
details = []

for h in HORIZONS:
    scores = {k: [] for k in ['mse', 'rmse', 'mae', 'mape']}
    for i, sid in enumerate(series_ids):
        store, product = sid.split('_', 1)
        r = rolling_eval(series_dict[sid], VAL_END, h)
        for k in scores:
            scores[k].append(r[k])
        details.append({'model': 'Prophet', 'store': store, 'product': product, 'horizon': h, **{k: round(float(v), 4) for k, v in r.items()}})
        if (i + 1) % 20 == 0:
            print(f'  H={h} | {i+1}/{len(series_ids)} done')
    row = {
        'model': 'Prophet', 'dataset': 'retail_inventory_daily', 'target': TARGET,
        'horizon': h,
        'mean_mse':  round(float(np.nanmean(scores['mse'])), 4),
        'mean_rmse': round(float(np.nanmean(scores['rmse'])), 4),
        'mean_mae':  round(float(np.nanmean(scores['mae'])), 4),
        'mean_mape': round(float(np.nanmean(scores['mape'])), 4),
    }
    results.append(row)
    print(f"H={h:2d} | MSE={row['mean_mse']:.2f} RMSE={row['mean_rmse']:.2f} MAE={row['mean_mae']:.2f} MAPE={row['mean_mape']:.2f}%")

pd.DataFrame(results).to_csv(f'{RESULT_DIR}/prophet_daily_summary.csv', index=False)
pd.DataFrame(details).to_csv(f'{RESULT_DIR}/prophet_daily_details.csv', index=False)
print(f'\nSaved to {RESULT_DIR}/prophet_daily_summary.csv')

02:32:23 - cmdstanpy - INFO - Chain [1] start processing
02:32:23 - cmdstanpy - INFO - Chain [1] done processing
02:32:24 - cmdstanpy - INFO - Chain [1] start processing
02:32:24 - cmdstanpy - INFO - Chain [1] done processing
02:32:24 - cmdstanpy - INFO - Chain [1] start processing
02:32:24 - cmdstanpy - INFO - Chain [1] done processing
02:32:24 - cmdstanpy - INFO - Chain [1] start processing
02:32:24 - cmdstanpy - INFO - Chain [1] done processing
02:32:24 - cmdstanpy - INFO - Chain [1] start processing
02:32:24 - cmdstanpy - INFO - Chain [1] done processing
02:32:24 - cmdstanpy - INFO - Chain [1] start processing
02:32:24 - cmdstanpy - INFO - Chain [1] done processing
02:32:24 - cmdstanpy - INFO - Chain [1] start processing
02:32:24 - cmdstanpy - INFO - Chain [1] done processing
02:32:25 - cmdstanpy - INFO - Chain [1] start processing
02:32:25 - cmdstanpy - INFO - Chain [1] done processing
02:32:25 - cmdstanpy - INFO - Chain [1] start processing
02:32:25 - cmdstanpy - INFO - Chain [1]

  H=7 | 20/100 done


02:32:50 - cmdstanpy - INFO - Chain [1] start processing
02:32:50 - cmdstanpy - INFO - Chain [1] done processing
02:32:50 - cmdstanpy - INFO - Chain [1] start processing
02:32:50 - cmdstanpy - INFO - Chain [1] done processing
02:32:50 - cmdstanpy - INFO - Chain [1] start processing
02:32:51 - cmdstanpy - INFO - Chain [1] done processing
02:32:51 - cmdstanpy - INFO - Chain [1] start processing
02:32:51 - cmdstanpy - INFO - Chain [1] done processing
02:32:51 - cmdstanpy - INFO - Chain [1] start processing
02:32:51 - cmdstanpy - INFO - Chain [1] done processing
02:32:51 - cmdstanpy - INFO - Chain [1] start processing
02:32:51 - cmdstanpy - INFO - Chain [1] done processing
02:32:51 - cmdstanpy - INFO - Chain [1] start processing
02:32:51 - cmdstanpy - INFO - Chain [1] done processing
02:32:51 - cmdstanpy - INFO - Chain [1] start processing
02:32:51 - cmdstanpy - INFO - Chain [1] done processing
02:32:51 - cmdstanpy - INFO - Chain [1] start processing
02:32:51 - cmdstanpy - INFO - Chain [1]

  H=7 | 40/100 done


02:33:16 - cmdstanpy - INFO - Chain [1] done processing
02:33:16 - cmdstanpy - INFO - Chain [1] start processing
02:33:16 - cmdstanpy - INFO - Chain [1] done processing
02:33:17 - cmdstanpy - INFO - Chain [1] start processing
02:33:17 - cmdstanpy - INFO - Chain [1] done processing
02:33:17 - cmdstanpy - INFO - Chain [1] start processing
02:33:17 - cmdstanpy - INFO - Chain [1] done processing
02:33:17 - cmdstanpy - INFO - Chain [1] start processing
02:33:17 - cmdstanpy - INFO - Chain [1] done processing
02:33:17 - cmdstanpy - INFO - Chain [1] start processing
02:33:17 - cmdstanpy - INFO - Chain [1] done processing
02:33:17 - cmdstanpy - INFO - Chain [1] start processing
02:33:17 - cmdstanpy - INFO - Chain [1] done processing
02:33:17 - cmdstanpy - INFO - Chain [1] start processing
02:33:17 - cmdstanpy - INFO - Chain [1] done processing
02:33:17 - cmdstanpy - INFO - Chain [1] start processing
02:33:17 - cmdstanpy - INFO - Chain [1] done processing
02:33:17 - cmdstanpy - INFO - Chain [1] 

...

  H=28 | 60/100 done


02:35:54 - cmdstanpy - INFO - Chain [1] start processing
02:35:54 - cmdstanpy - INFO - Chain [1] done processing
02:35:55 - cmdstanpy - INFO - Chain [1] start processing
02:35:55 - cmdstanpy - INFO - Chain [1] done processing
02:35:55 - cmdstanpy - INFO - Chain [1] start processing
02:35:55 - cmdstanpy - INFO - Chain [1] done processing
02:35:55 - cmdstanpy - INFO - Chain [1] start processing
02:35:55 - cmdstanpy - INFO - Chain [1] done processing
02:35:55 - cmdstanpy - INFO - Chain [1] start processing
02:35:55 - cmdstanpy - INFO - Chain [1] done processing
02:35:55 - cmdstanpy - INFO - Chain [1] start processing
02:35:55 - cmdstanpy - INFO - Chain [1] done processing
02:35:55 - cmdstanpy - INFO - Chain [1] start processing
02:35:55 - cmdstanpy - INFO - Chain [1] done processing
02:35:55 - cmdstanpy - INFO - Chain [1] start processing
02:35:55 - cmdstanpy - INFO - Chain [1] done processing
02:35:55 - cmdstanpy - INFO - Chain [1] start processing
02:35:55 - cmdstanpy - INFO - Chain [1]

  H=28 | 80/100 done


02:36:01 - cmdstanpy - INFO - Chain [1] start processing
02:36:01 - cmdstanpy - INFO - Chain [1] done processing
02:36:01 - cmdstanpy - INFO - Chain [1] start processing
02:36:01 - cmdstanpy - INFO - Chain [1] done processing
02:36:01 - cmdstanpy - INFO - Chain [1] start processing
02:36:01 - cmdstanpy - INFO - Chain [1] done processing
02:36:01 - cmdstanpy - INFO - Chain [1] start processing
02:36:01 - cmdstanpy - INFO - Chain [1] done processing
02:36:01 - cmdstanpy - INFO - Chain [1] start processing
02:36:01 - cmdstanpy - INFO - Chain [1] done processing
02:36:01 - cmdstanpy - INFO - Chain [1] start processing
02:36:01 - cmdstanpy - INFO - Chain [1] done processing
02:36:01 - cmdstanpy - INFO - Chain [1] start processing
02:36:01 - cmdstanpy - INFO - Chain [1] done processing
02:36:01 - cmdstanpy - INFO - Chain [1] start processing
02:36:01 - cmdstanpy - INFO - Chain [1] done processing
02:36:02 - cmdstanpy - INFO - Chain [1] start processing
02:36:02 - cmdstanpy - INFO - Chain [1]

  H=28 | 100/100 done
H=28 | MSE=1641.04 RMSE=39.97 MAE=31.75 MAPE=4576698607.90%

Saved to /kaggle/working/prophet_daily_summary.csv


In [8]:
pd.DataFrame(results)

,model,dataset,target,horizon,mean_mse,mean_rmse,mean_mae,mean_mape
0,Prophet,retail_inventory_daily,Units Sold,7,1610.2865,39.5931,31.3358,4.554659e+09
1,Prophet,retail_inventory_daily,Units Sold,14,1636.2751,39.9215,31.6921,4.534711e+09
2,Prophet,retail_inventory_daily,Units Sold,28,1641.0446,39.9726,31.7497,4.576699e+09
